# Mastering Stacking
Welcome to the guide on Stacking.
### Kaggle Dataset:
[Titanic Dataset](https://www.kaggle.com/c/titanic)


### Why and What: Imports
**WHY:** Libraries for data manipulation.
**WHAT:** pandas, numpy, sklearn, matplotlib.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, roc_curve, auc
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
sns.set_theme(style='whitegrid')


## Part 1: Theory Recap

Stacking is the ultimate ensemble technique, combining entirely different model architectures.

### Real-World Analogy
You are a CEO (Meta-Learner) with three advisors: a Data Analyst, a Lawyer, and an Accountant (Base Models). You don't just take a majority vote; you learn that the Accountant is always right about budget issues, and the Lawyer is right about contracts. You use machine learning to learn *how* to weigh their opinions.

### Core Mathematics

1. **Base Model Out-of-Fold Predictions (Meta-Features):**
$$ Z_{i, m} = \hat{f}_{m}^{-k(i)}(x_i) $$
* $Z_{i, m}$ : The prediction for instance $i$ generated by base model $m$.
* $\hat{f}_{m}^{-k(i)}$ : Model $m$ trained on all data folds *except* fold $k$ (which contains instance $i$).
* *Why?* If we don't use K-Fold, the base models will perfectly predict the training data, and the meta-learner will just overfit to the most complex base model.

2. **Meta-Learner Prediction:**
$$ \hat{y}_i = \text{MetaModel}(Z_{i, 1}, Z_{i, 2}, \dots, Z_{i, M}) $$
* $\hat{y}_i$ : Final stacked prediction.
* $\text{MetaModel}$ : Typically a simple linear model like Logistic Regression or Ridge Regression to prevent overfitting on the meta-features.


### Why and What: Loading Data
**WHY:** Real world data.
**WHAT:** Titanic.


In [ ]:
df = pd.read_csv('https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv')
display(df.head())


### Why and What: Preprocessing
**WHY:** ML models need clean numerical data.
**WHAT:** Fillna, encode, scale.


In [ ]:
df_clean = df.drop(['PassengerId', 'Name', 'Ticket', 'Cabin'], axis=1)
df_clean['Age'] = df_clean['Age'].fillna(df_clean['Age'].median())
df_clean['Fare'] = df_clean['Fare'].fillna(df_clean['Fare'].median())
df_clean['Embarked'] = df_clean['Embarked'].fillna(df_clean['Embarked'].mode()[0])
df_clean['Sex'] = df_clean['Sex'].map({'male': 0, 'female': 1})
df_clean = pd.get_dummies(df_clean, columns=['Embarked'], drop_first=True)
X = df_clean.drop('Survived', axis=1).values
y = df_clean['Survived'].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)


## Part 2: From Scratch Implementation
### Why and What: Scratch
**WHY:** Demystify the black box.
**WHAT:** Numpy implementation.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
class ScratchModel:
    def __init__(self):
        self.bases = [LogisticRegression(), KNeighborsClassifier(), DecisionTreeClassifier(max_depth=3)]
        self.meta = LogisticRegression()
    def fit(self, X, y):
        kf = KFold(n_splits=3, shuffle=True, random_state=42)
        oof = np.zeros((X.shape[0], len(self.bases)))
        for i, m in enumerate(self.bases):
            for tr_idx, val_idx in kf.split(X):
                m.fit(X[tr_idx], y[tr_idx])
                oof[val_idx, i] = m.predict_proba(X[val_idx])[:, 1]
            m.fit(X, y)
        self.meta.fit(oof, y)
    def predict_proba(self, X):
        meta_X = np.zeros((X.shape[0], len(self.bases)))
        for i, m in enumerate(self.bases):
            meta_X[:, i] = m.predict_proba(X)[:, 1]
        return self.meta.predict_proba(meta_X)
    def predict(self, X): return (self.predict_proba(X)[:,1] >= 0.5).astype(int)


### Why and What: Evaluation
**WHY:** Verify correctness.
**WHAT:** Predict and accuracy.


In [ ]:
scratch_model = ScratchModel()
scratch_model.fit(X_train, y_train)
y_pred_custom = scratch_model.predict(X_test)
print(f"Scratch Accuracy: {accuracy_score(y_test, y_pred_custom)*100:.2f}%")


## Part 3: Library Implementation
### Why and What: Library
**WHY:** Optimized production code.
**WHAT:** Official package.


In [ ]:
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
ests = [('lr', LogisticRegression()), ('knn', KNeighborsClassifier()), ('dt', DecisionTreeClassifier(max_depth=3))]
model_library = StackingClassifier(estimators=ests, final_estimator=LogisticRegression(), cv=3)
model_library.fit(X_train, y_train)
print('Stacking Accuracy:', accuracy_score(y_test, model_library.predict(X_test)))


### Why and What: Visualizations
**WHY:** Diagnose behavior.
**WHAT:** ROC and Importances.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
if hasattr(model_library, 'predict_proba'):
    fpr, tpr, _ = roc_curve(y_test, model_library.predict_proba(X_test)[:, 1])
    axes[0].plot(fpr, tpr, color='darkorange', label=f'ROC (area = {auc(fpr, tpr):.2f})')
axes[0].plot([0,1], [0,1], 'k--', label='Random Guess')
axes[0].set_title('ROC Curve')
axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')
axes[0].legend(loc='lower right')
importances = getattr(model_library, 'feature_importances_', None)
if importances is not None:
    idx = np.argsort(importances)
    axes[1].barh(range(len(idx)), importances[idx], color='teal', label='Importance')
    axes[1].set_yticks(range(len(idx)))
    axes[1].set_yticklabels(df_clean.drop('Survived', axis=1).columns[idx])
    axes[1].legend(loc='lower right')
axes[1].set_title('Feature Importances')
axes[1].set_xlabel('Relative Importance'); axes[1].set_ylabel('Features')
plt.tight_layout()
plt.show()


## Part 4: Hyperparameter Experiments
### Why and What: Tuning
**WHY:** Optimize model.
**WHAT:** Grid search.


In [ ]:
res=[]
for cv in [2, 5]:
 for meta in ['LogisticRegression', 'DecisionTreeClassifier']:
  f_est = LogisticRegression() if meta=='LogisticRegression' else DecisionTreeClassifier(max_depth=2)
  clf=StackingClassifier(estimators=ests, final_estimator=f_est, cv=cv)
  res.append({'cv':cv, 'meta':meta, 'acc':cross_val_score(clf, X_scaled, y, cv=3).mean()})
sns.barplot(data=pd.DataFrame(res), x='cv', y='acc', hue='meta')
plt.title('Stacking Hyperparameters'); plt.legend(title='Meta'); plt.show()


## Part 5: Interview Corner
**Q: Why use Out-of-Fold predictions instead of training data predictions?**
**A:** If we feed predictions from models on the data they trained on, highly overfit models will pass 100% accuracy to the meta-model, causing target leakage.


## Key Takeaways
- Base models.
- Meta model.
- OOF predictions.
- Diversity is key.
- Kaggle winning technique.
